# HW3 - Seeded Deutsch-Jozsa Oracle Classification

In this assignment, you will implement a personalized Deutsch-Jozsa circuit.
Your seed generates an oracle of the form:

`f(x) = mask · x XOR output_offset`

where `mask` is given in `q[0], q[1], ..., q[n-1]` order.

Your job is to:
1. classify the oracle as **constant** or **balanced** before building the circuit,
2. build the seeded oracle,
3. run the Deutsch-Jozsa circuit on the ideal simulator,
4. interpret the displayed bitstring using the measurement map,
5. optionally run a smaller version on IBM hardware,
6. export `answers.json`.

Do not hard-code an answer. Your output must come from your own seeded circuit execution.

> **Completed worked example version.** This copy fills in the student TODO cells for demonstration/testing. For an actual student-facing release, use the template version and keep the instructor/reference logic hidden.

In [ ]:
%pip -q install qiskit qiskit-aer matplotlib jsonschema

In [ ]:
import json, math, hashlib
from typing import List, Dict, Any
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

## 1. Student ID and seed

Replace `demo_student` with your assigned student ID. Do not change it after generating results.

In [ ]:
STUDENT_ID = "demo_student"
ASSIGNMENT_ID = "HW3"
SHOTS = 2048
HARDWARE_SHOTS = 4096

## 2. Generate your seeded oracle configuration

This function gives you the assignment configuration. It gives you the mask and measurement map, but it does not give you the expected measured result.

Important: `mask_q0_to_qn` is listed in `q[0]...q[n-1]` order.

In [ ]:
def stable_seed(student_id: str, assignment_id: str = "HW3") -> int:
    key = f"{assignment_id}|{student_id}".encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:12], 16)


def bits_from_seed(seed: int, n: int, offset: int = 0):
    return [(seed >> (offset + i)) & 1 for i in range(n)]


def is_palindrome(bits):
    return bits == list(reversed(bits))


def make_nonzero_nonpal_mask(seed: int, n: int):
    for shift in range(0, 40, 3):
        mask = bits_from_seed(seed, n, shift)
        if any(mask) and not is_palindrome(mask):
            return mask
    mask = [0] * n
    mask[0] = 1
    mask[-2] = 1
    if is_palindrome(mask):
        mask[1] ^= 1
    return mask


def generate_config(student_id: str, assignment_id: str = "HW3"):
    seed = stable_seed(student_id, assignment_id)
    n = 3 + (seed % 3)
    make_constant = ((seed >> 5) % 3 == 0)
    if make_constant:
        mask = [0] * n
    else:
        mask = make_nonzero_nonpal_mask(seed >> 8, n)
    output_offset = (seed >> 17) & 1

    measurement_mode = "reversed" if ((seed >> 19) & 1) else "direct"
    if measurement_mode == "direct":
        measurement_map = [[i, i] for i in range(n)]
    else:
        measurement_map = [[i, n - 1 - i] for i in range(n)]

    hw_n = 3
    hw_make_constant = ((seed >> 23) % 4 == 0)
    if hw_make_constant:
        hw_mask = [0] * hw_n
    else:
        hw_mask = make_nonzero_nonpal_mask(seed >> 25, hw_n)
    hw_output_offset = (seed >> 31) & 1
    hw_measurement_mode = "reversed" if ((seed >> 32) & 1) else "direct"
    if hw_measurement_mode == "direct":
        hw_measurement_map = [[i, i] for i in range(hw_n)]
    else:
        hw_measurement_map = [[i, hw_n - 1 - i] for i in range(hw_n)]

    return {
        "assignment_id": assignment_id,
        "student_id": student_id,
        "seed": seed,
        "num_input_qubits": n,
        "mask_q0_to_qn": mask,
        "output_offset": output_offset,
        "measurement_mode": measurement_mode,
        "measurement_map": measurement_map,
        "hardware_num_input_qubits": hw_n,
        "hardware_mask_q0_to_qn": hw_mask,
        "hardware_output_offset": hw_output_offset,
        "hardware_measurement_mode": hw_measurement_mode,
        "hardware_measurement_map": hw_measurement_map,
    }

config = generate_config(STUDENT_ID, ASSIGNMENT_ID)
print(json.dumps(config, indent=2))

## 3. Classify the oracle before building it

Use the mask to decide whether your oracle is constant or balanced.

Rules:
- If every mask bit is 0, then the function does not depend on the input. It is **constant**.
- If at least one mask bit is 1, then the function is a parity function over selected input bits. It is **balanced**.

`output_offset` may flip every output value, but it does not change whether the oracle is constant or balanced.

In [ ]:
# Classify the oracle before building the circuit.
# For f(x) = mask · x XOR output_offset:
# - all-zero mask  -> constant function
# - nonzero mask   -> balanced parity function
# output_offset flips all outputs but does not change constant/balanced classification.

predicted_oracle_classification = "constant" if not any(config["mask_q0_to_qn"]) else "balanced"

print("Mask q[0]...q[n-1]:", config["mask_q0_to_qn"])
print("Output offset:", config["output_offset"])
print("My oracle classification:", predicted_oracle_classification)

## 4. Build the seeded oracle

For the oracle `f(x) = mask · x XOR output_offset`:
- If `output_offset == 1`, apply X to the output qubit.
- For each mask bit equal to 1, apply CX from that input qubit to the output qubit.

Do not measure inside the oracle.

In [ ]:
def build_oracle(qc, input_qubits, output_qubit, mask_q0_to_qn, output_offset):
    """Build U_f for f(x) = mask · x XOR output_offset.

    The oracle acts as |x, y> -> |x, y XOR f(x)>.
    - output_offset == 1 flips the output for every input.
    - mask[i] == 1 means input qubit i contributes to the parity, so apply CX(input[i], output).
    """
    if output_offset == 1:
        qc.x(output_qubit)

    for i, bit in enumerate(mask_q0_to_qn):
        if bit == 1:
            qc.cx(input_qubits[i], output_qubit)

    return qc

## 5. Build the full Deutsch-Jozsa circuit

Circuit structure:
1. Prepare the output/ancilla qubit in `|1>`.
2. Apply H to all qubits, so the output becomes `|->` and inputs become a superposition.
3. Apply the oracle.
4. Apply H to input qubits only.
5. Measure input qubits only using the seeded measurement map.

The output/ancilla qubit is not measured for the main answer.

In [ ]:
def build_deutsch_jozsa_circuit(config):
    n = config["num_input_qubits"]
    q = QuantumRegister(n + 1, "q")
    c = ClassicalRegister(n, "c")
    qc = QuantumCircuit(q, c)
    output = q[n]

    # 1. Prepare the output/ancilla qubit in |1>, then H makes it |->.
    qc.x(output)
    qc.h(output)

    # 2. Put all input qubits into superposition.
    for i in range(n):
        qc.h(q[i])

    qc.barrier()

    # 3. Apply the seeded oracle.
    build_oracle(
        qc,
        [q[i] for i in range(n)],
        output,
        config["mask_q0_to_qn"],
        config["output_offset"],
    )

    qc.barrier()

    # 4. Apply H to the input register again for the Deutsch-Jozsa interference step.
    for i in range(n):
        qc.h(q[i])

    qc.barrier()

    # 5. Measure input qubits only, using the seeded measurement map.
    # The output/ancilla qubit is intentionally not measured for the main answer.
    for q_index, c_index in config["measurement_map"]:
        qc.measure(q[q_index], c[c_index])

    return qc

qc = build_deutsch_jozsa_circuit(config)
qc.draw("mpl")

## 6. Predict the expected bitstring

Do this before looking at simulator counts.

Rules:
- If the oracle is constant, the input-register result should be all zeros.
- If the oracle is balanced, the logical input-register result should match the mask.
- The logical string is written as `q[n-1]...q[0]`.
- The displayed Qiskit count string is written as `c[n-1]...c[0]`, using your measurement map.

In [ ]:
def expected_input_bits_q0_to_qn(config):
    """Expected Deutsch-Jozsa input-register result in q[0]...q[n-1] order."""
    mask = config["mask_q0_to_qn"]

    # Constant oracle -> all zeros.
    # Balanced parity oracle -> output input-register pattern matches the mask.
    if not any(mask):
        return [0] * len(mask)
    return list(mask)


def displayed_bitstring_from_bits(bits_q0_to_qn, measurement_map):
    """Convert logical input-register bits into Qiskit's displayed c[n-1]...c[0] count string."""
    n = len(bits_q0_to_qn)
    classical_bits = [0] * n

    # Fill classical bits according to the measurement map.
    for q_index, c_index in measurement_map:
        classical_bits[c_index] = bits_q0_to_qn[q_index]

    # Qiskit displays classical result strings as c[n-1]...c[0].
    return "".join(str(classical_bits[i]) for i in reversed(range(n)))


expected_bits_q0_to_qn = expected_input_bits_q0_to_qn(config)
expected_logical = "".join(str(expected_bits_q0_to_qn[i]) for i in reversed(range(len(expected_bits_q0_to_qn))))
expected_display = displayed_bitstring_from_bits(expected_bits_q0_to_qn, config["measurement_map"])

print("Expected input bits q[0]...q[n-1]:", expected_bits_q0_to_qn)
print("Expected logical qubit string q[n-1]...q[0]:", expected_logical)
print("Expected displayed count string c[n-1]...c[0]:", expected_display)

## 7. Run the ideal simulator

The simulator should be strongly dominated by the expected displayed bitstring. For this exact Deutsch-Jozsa construction, it should usually be all shots on one bitstring.

In [ ]:
sim = AerSimulator(seed_simulator=config["seed"] % (2**32 - 1))
tqc = transpile(qc, sim, seed_transpiler=config["seed"] % (2**32 - 1))
result = sim.run(tqc, shots=SHOTS).result()
simulator_counts = {str(k): int(v) for k, v in result.get_counts().items()}
dominant_bitstring = max(simulator_counts, key=simulator_counts.get)

print("Simulator counts:", simulator_counts)
print("Dominant bitstring:", dominant_bitstring)
print("Expected displayed bitstring:", expected_display)
print("Match?", dominant_bitstring == expected_display)
plot_histogram(simulator_counts, title="HW3 Deutsch-Jozsa simulator counts")

## 8. Record circuit metrics

These metrics help detect missing gates, accidental shortcut circuits, and transpiler-related changes.

In [ ]:
original_depth = qc.depth()
transpiled_depth_simulator = tqc.depth()
operation_counts_simulator = {str(k): int(v) for k, v in tqc.count_ops().items()}

print("Original depth:", original_depth)
print("Transpiled simulator depth:", transpiled_depth_simulator)
print("Operation counts:", operation_counts_simulator)

## 9. Optional IBM hardware extension

This section is optional/research-oriented. Run it only if you have IBM Quantum access.

The hardware task uses a smaller 3-input-qubit seeded Deutsch-Jozsa circuit to reduce queue time and noise.

In [ ]:
# Optional. Uncomment and run if you have IBM Quantum access.
# %pip -q install qiskit-ibm-runtime

In [ ]:
# Optional IBM hardware code.
# Leave RUN_IBM_HARDWARE = False if you are only completing the autograded simulator core.

RUN_IBM_HARDWARE = False

hardware_run_completed = False
hardware_backend = None
hardware_job_id = None
hardware_counts = {}
hardware_dominant_bitstring = None
hardware_expected_percentage = None
hardware_result_classification = None

def build_hardware_config(config):
    """Use the smaller 3-input-qubit hardware version from the same seed."""
    hw_config = dict(config)
    hw_config["num_input_qubits"] = config["hardware_num_input_qubits"]
    hw_config["mask_q0_to_qn"] = config["hardware_mask_q0_to_qn"]
    hw_config["output_offset"] = config["hardware_output_offset"]
    hw_config["measurement_mode"] = config["hardware_measurement_mode"]
    hw_config["measurement_map"] = config["hardware_measurement_map"]
    return hw_config

def classify_result_from_percentage(expected_percentage, dominant_bitstring, expected_bitstring):
    if expected_percentage >= 70:
        return "successful"
    if expected_percentage >= 40:
        return "partial"
    if dominant_bitstring == expected_bitstring:
        return "inconclusive"
    return "failed"

if RUN_IBM_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    hw_config = build_hardware_config(config)
    hw_qc = build_deutsch_jozsa_circuit(hw_config)

    hw_expected_bits_q0_to_qn = expected_input_bits_q0_to_qn(hw_config)
    hardware_expected_display = displayed_bitstring_from_bits(
        hw_expected_bits_q0_to_qn,
        hw_config["measurement_map"],
    )

    service = QiskitRuntimeService(channel="ibm_quantum")
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=4)
    hardware_backend = backend.name

    pm = generate_preset_pass_manager(
        backend=backend,
        optimization_level=1,
        seed_transpiler=config["seed"] % (2**32 - 1),
    )
    isa_circuit = pm.run(hw_qc)

    sampler = Sampler(backend)
    job = sampler.run([isa_circuit], shots=HARDWARE_SHOTS)
    hardware_job_id = job.job_id()
    print("Backend:", hardware_backend)
    print("Job ID:", hardware_job_id)
    print("Initial status:", job.status())

    result = job.result()
    pub_result = result[0]

    try:
        extracted_counts = pub_result.data.c.get_counts()
    except Exception:
        extracted_counts = None
        for field_name in dir(pub_result.data):
            if field_name.startswith("_"):
                continue
            field = getattr(pub_result.data, field_name)
            if hasattr(field, "get_counts"):
                extracted_counts = field.get_counts()
                print("Counts extracted from classical register:", field_name)
                break
        if extracted_counts is None:
            raise RuntimeError("Could not extract counts from SamplerV2 result.")

    hardware_counts = {str(k): int(v) for k, v in extracted_counts.items()}
    hardware_dominant_bitstring = max(hardware_counts, key=hardware_counts.get)
    hardware_expected_percentage = 100 * hardware_counts.get(hardware_expected_display, 0) / HARDWARE_SHOTS
    hardware_result_classification = classify_result_from_percentage(
        hardware_expected_percentage,
        hardware_dominant_bitstring,
        hardware_expected_display,
    )
    hardware_run_completed = True

    print("Expected hardware display string:", hardware_expected_display)
    print("Hardware counts:", hardware_counts)
    print("Dominant:", hardware_dominant_bitstring)
    print("Expected percentage:", hardware_expected_percentage)
    print("Hardware result classification:", hardware_result_classification)
    plot_histogram(hardware_counts, title=f"IBM hardware Deutsch-Jozsa on {hardware_backend}")
else:
    print("IBM hardware section skipped. The simulator core remains complete.")

## 10. Reflection questions

Answer each in 2-4 sentences.

**Q1. Oracle classification:** How did you classify your oracle as constant or balanced before building it? How did the mask and output offset affect that decision?

**Q2. Bit ordering:** How did the measurement map affect the displayed Qiskit count string? Why can `c[n-1]...c[0]` differ from the mask written in `q[0]...q[n-1]` order?

**Q3. Hardware noise:** If you ran hardware, compare your hardware counts to the simulator counts. If you did not run hardware, explain what noise effects you would expect and why.

In [ ]:
reflection_oracle_classification = f"""
I classified the oracle by checking the mask {config['mask_q0_to_qn']}. If the mask is all zeros, the function does not depend on the input bits, so the oracle is constant. If at least one mask bit is 1, the function is a parity function over selected input bits, so it is balanced. The output_offset value {config['output_offset']} can flip all outputs, but it does not change whether the function is constant or balanced.
""".strip()

reflection_bit_order = f"""
The mask is written in q[0]...q[n-1] order, but Qiskit displays measurement count strings in c[n-1]...c[0] order. I used the measurement_map {config['measurement_map']} to place each measured qubit value into the correct classical bit before reversing the classical bits for display. This is why the displayed count string can differ from the logical mask order.
""".strip()

if hardware_run_completed:
    reflection_hardware_noise = f"""
The IBM hardware run used backend {hardware_backend} and produced dominant bitstring {hardware_dominant_bitstring}. The expected bitstring appeared with {hardware_expected_percentage:.2f}% of shots, so I classified the hardware result as {hardware_result_classification}. Any extra bitstrings are expected because real quantum devices have gate error, readout error, routing overhead, and decoherence.
""".strip()
else:
    reflection_hardware_noise = """
I did not run the optional IBM hardware section for this submission. On real hardware, I would expect the simulator's single clean output to spread across additional bitstrings because of gate errors, readout errors, routing overhead after transpilation, and decoherence. A successful hardware result would still have the expected Deutsch-Jozsa bitstring as the dominant output.
""".strip()

print("Reflection 1:", reflection_oracle_classification)
print("Reflection 2:", reflection_bit_order)
print("Reflection 3:", reflection_hardware_noise)

## 11. Export answers.json

Submit this file. The hidden autograder will regenerate your reference output from your student ID and compare it with your JSON.

In [ ]:
answers = {
    "assignment_id": ASSIGNMENT_ID,
    "student_id": STUDENT_ID,
    "seed": config["seed"],
    "num_input_qubits": config["num_input_qubits"],
    "shots": SHOTS,
    "mask_q0_to_qn": config["mask_q0_to_qn"],
    "output_offset": config["output_offset"],
    "measurement_mode": config["measurement_mode"],
    "measurement_map": config["measurement_map"],
    "predicted_oracle_classification": predicted_oracle_classification,
    "expected_logical_qn_to_q0": expected_logical,
    "expected_display_bitstring": expected_display,
    "simulator_counts": simulator_counts,
    "dominant_bitstring": dominant_bitstring,
    "original_depth": original_depth,
    "transpiled_depth_simulator": transpiled_depth_simulator,
    "operation_counts_simulator": operation_counts_simulator,
    "hardware_run_completed": hardware_run_completed,
    "hardware_backend": hardware_backend,
    "hardware_job_id": hardware_job_id,
    "hardware_counts": hardware_counts,
    "hardware_dominant_bitstring": hardware_dominant_bitstring,
    "hardware_expected_percentage": hardware_expected_percentage,
    "hardware_result_classification": hardware_result_classification,
    "reflection_oracle_classification": reflection_oracle_classification,
    "reflection_bit_order": reflection_bit_order,
    "reflection_hardware_noise": reflection_hardware_noise,
}

with open("answers.json", "w", encoding="utf-8") as f:
    json.dump(answers, f, indent=2)

print(json.dumps(answers, indent=2))
print("Saved answers.json")